# Paired Vinamilk reranker benchmark on Kaggle GPU

Compare BGE Reranker v2 m3 and Qwen3-Reranker-0.6B on the exact same 12 role-filtered dense candidate sets (up to 30 passages/query). The private attached dataset contains public-source excerpts only, frozen case labels, and a database-free evaluator bundle. It has no credentials or private uploads.

Enable Internet and a GPU accelerator. Each model runs in a separate process with identical candidate SHA, case SHA, max sequence length, batch size, and device. This remains a small, non-exhaustively labeled thesis benchmark; do not treat one run as a production winner.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before running this comparison."
print(torch.cuda.get_device_name(0))

inputs = Path("/kaggle/input")
bundles = list(inputs.rglob("reranker_candidates_top30.json"))
project_scripts = list(inputs.rglob("scripts.retrieval.evaluate_rerankers.py"))
print("Attached files:", [str(path.relative_to(inputs)) for path in inputs.rglob("*") if path.is_file()])
assert len(bundles) == len(project_scripts) == 1, "Attach exactly one vinamilk-reranker-top30-benchmark-assets dataset."
work = Path("/kaggle/working/reranker-benchmark")
project = work / "project"
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
project_source = project_scripts[0].parent.parent
shutil.copytree(project_source, project)
assert (project / "scripts/retrieval/evaluate_rerankers.py").is_file()
candidate_bundle = project / "data/evaluation/vinamilk-reranker-candidates-top30.json"
candidate_bundle.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(bundles[0], candidate_bundle)

bundle = json.loads(candidate_bundle.read_text(encoding="utf-8"))
cases_path = project / "data/evaluation/archive/retrieval/vinamilk_retrieval_cases.json"
case_sha = hashlib.sha256(cases_path.read_bytes()).hexdigest()
assert bundle["case_file_sha256"] == case_sha, "Candidate bundle and judgment labels do not match."
print({"case_count": len(bundle["cases"]), "candidate_sha256": bundle["candidate_sha256"], "case_file_sha256": case_sha, "candidate_k": bundle["candidate_k"]})

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers==6.0.1", "python-dotenv==1.1.1"], check=True)
print("Pinned evaluator dependencies installed.")

In [ ]:
results_dir = Path("/kaggle/working/reranker-results")
results_dir.mkdir(parents=True, exist_ok=True)
models = ["bge-reranker-v2-m3", "qwen3-reranker-0.6b"]
reports = {}
for model in models:
    output = results_dir / f"{model}.json"
    command = [
        sys.executable, "scripts/retrieval/evaluate_rerankers.py",
        "--model", model,
        "--candidate-input", str(candidate_bundle),
        "--candidate-k", "30",
        "--device", "cuda",
        "--batch-size", "8",
        "--max-sequence-length", "256",
        "--output", str(output),
    ]
    subprocess.run(command, cwd=project, check=True)
    reports[model] = json.loads(output.read_text(encoding="utf-8"))

assert len({r["candidate_sha256"] for r in reports.values()}) == 1
assert len({r["case_file_sha256"] for r in reports.values()}) == 1
assert all(r["environment"]["device"] == "cuda" for r in reports.values())
for model, report in reports.items():
    print(model, {
        "model_revision": report["model_revision"],
        "candidate_sha256": report["candidate_sha256"],
        "case_count": report["case_count"],
        "passage_dense_mrr": report["passage_aggregate"]["dense"]["mrr"],
        "passage_reranked_mrr": report["passage_aggregate"]["reranked"]["mrr"],
        "passage_dense_recall_at_5": report["passage_aggregate"]["dense"].get("recall@5"),
        "passage_reranked_recall_at_5": report["passage_aggregate"]["reranked"].get("recall@5"),
        "mean_rerank_seconds": report["timing_seconds"]["mean_rerank"],
        "peak_rss_mb": report["environment"]["peak_rss_mb"],
        "gpu": report["environment"].get("gpu_name"),
    })

## Result files and limitations

Download both JSON reports from `/kaggle/working/reranker-results/` and keep them together. They record the frozen case/candidate hashes, model revisions, per-case before/after ranking, latency, device, and memory. The candidates are already role-filtered on the source host; no database URL, Supabase key, or private upload is sent to Kaggle. Passage labels are selected and non-exhaustive, candidate pools have at most 30 passages, and this measures ranking only—not answer quality or user benefit. Compare downstream QA and expand independent judgments before selecting a model.